## I. Các gói thư viện sử dụng

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [23]:
!pip install python-dotenv huggingface_hub datasets -q

In [24]:
import os
import re

from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
import numpy as np
import pandas as pd
from sklearn.feature_selection import f_classif, mutual_info_classif

## II. Kết nối môi trường

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [29]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [30]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [31]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

### Đánh giá chất lượng

In [42]:
class DataQualityValidator:
    def __init__(self, df, label_col='Label', text_cols=None):
        self.df = df.copy()
        self.label_col = label_col
        self.text_cols = text_cols if text_cols else ['Text', 'segmented_text']
        self.numeric_cols = [col for col in self.df.select_dtypes(include=[np.number]).columns
                             if col != self.label_col]


    def check_completeness(self):
        null_counts = self.df.isnull().sum()
        total_null = null_counts.sum()
        if total_null != 0:
            print("Phát hiện giá trị Null/NaN:")
            print(null_counts[null_counts > 0])

    def check_consistency(self):
        for col in self.text_cols:
            if col in self.df.columns:
                is_lower = self.df[col].str.islower().all()
                print(f"Cột '{col}' {'viết thường' if is_lower else 'viết hoa'}")

    def check_variance_threshold(self):
        variances = self.df[self.numeric_cols].var()
        zero_var_cols = variances[variances == 0].index.tolist()
        if zero_var_cols:
            print(f"Các cột vô giá trị (Variance = 0) cần loại bỏ: {zero_var_cols}")

        return zero_var_cols

    def check_feature_power(self):
        X = self.df[self.numeric_cols].fillna(0)
        y = self.df[self.label_col]
        f_scores, p_values = f_classif(X, y)
        mi_scores = mutual_info_classif(X, y)

        stats_df = pd.DataFrame({
            'Feature': self.numeric_cols,
            'F-Score': f_scores,
            'P-Value': p_values,
            'Info_Gain (MI)': mi_scores
        }).sort_values(by='Info_Gain (MI)', ascending=False)

        print(stats_df.to_string(index=False))
        return stats_df

    def check_correlation(self, threshold=0.85):
        corr_matrix = self.df[self.numeric_cols].corr(method='pearson')

        highly_correlated = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i):
                if abs(corr_matrix.iloc[i, j]) > threshold:
                    highly_correlated.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

        if highly_correlated:
            print("Các cặp đặc trưng sau có độ tương quan quá cao")
            for feat1, feat2, corr_val in highly_correlated:
                print(f"   - {feat1} & {feat2}: {corr_val:.4f}")

    def check_imbalance(self):
        dist = self.df[self.label_col].value_counts()
        dist_pct = self.df[self.label_col].value_counts(normalize=True) * 100

        for label, count in dist.items():
            print(f"Nhãn {label}: {count} dòng ({dist_pct[label]:.2f}%)")

        if abs(dist_pct.iloc[0] - dist_pct.iloc[-1]) > 20:
            print("Tập dữ liệu đang bị mất cân bằng!")
        else:
            print("Phân phối nhãn tương đối cân bằng.")

    def check_leakage(self, test_df):
        train_texts = set(self.df['Text'].dropna())
        test_texts = set(test_df['Text'].dropna())
        overlap = train_texts.intersection(test_texts)

        if len(overlap) != 0:
            print(f"Phát hiện {len(overlap)} dòng dữ liệu bị rò rỉ từ Train sang Test!")


    def verify_dataset(self, test_df=None):
        print(f"Cỡ mẫu {self.df.shape[0]} dòng, {self.df.shape[1]} cột")
        print("-"*60)

        self.check_completeness()
        self.check_consistency()

        self.check_variance_threshold()
        print("\n")
        self.check_feature_power()
        print("\n")
        self.check_correlation()

        self.check_imbalance()
        if test_df is not None:
            self.check_leakage(test_df)


In [43]:
dataset = load_dataset("JuniorThanh/vietnamese_news_human_ai_cleaned")

train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas() if 'test' in dataset else None

validator = DataQualityValidator(train_df, label_col='Label')
validator.verify_dataset(test_df=test_df)

Cỡ mẫu 7000 dòng, 14 cột
------------------------------------------------------------
Cột 'Text' viết hoa
Cột 'segmented_text' viết thường


           Feature     F-Score       P-Value  Info_Gain (MI)
           entropy 1434.585039 9.794214e-286        0.194731
special_char_ratio 1072.943929 4.495582e-219        0.123180
        burstiness 1129.314239 1.165991e-229        0.114615
 conjunction_ratio 1309.280591 5.721491e-263        0.101288
   stopwords_ratio  555.572798 2.843547e-118        0.085086
        caps_ratio 1116.876833 2.488947e-227        0.076764
      number_ratio  747.946906 1.508480e-156        0.068152
    sentence_count  123.828197  1.592435e-28        0.066332
         adj_ratio  707.395457 1.464647e-148        0.057801
               ttr   95.893031  1.690601e-22        0.054957
     pronoun_ratio    8.413621  3.735803e-03        0.036428


Nhãn 1: 3500 dòng (50.00%)
Nhãn 0: 3500 dòng (50.00%)
Phân phối nhãn tương đối cân bằng.
